# Part 1: Cloud Microphysics Foundations
## Interactive Learning with PyLCM

### Learning Objectives
- Understand adiabatic parcel theory and the lifting condensation level (LCL)
- Describe how aerosol size distributions and Köhler theory control CCN activation
- Explain the diffusional growth equation and the supersaturation budget
- Analyze collision-coalescence and the autoconversion bottleneck
- Run and interpret a full warm rain simulation

### Prerequisites
- Thermodynamics: ideal gas law, first law, Clausius-Clapeyron
- Python: NumPy arrays, matplotlib plotting
- Jupyter: cell execution, keyboard shortcuts

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
import copy, warnings, time
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '.')

from PyLCM.parameters import *
from PyLCM.micro_particle import *
from PyLCM.aero_init import aero_init, lognormal_pdf, r_equi
from PyLCM.parcel import ascend_parcel, parcel_rho, create_env_profiles
from PyLCM.condensation import esatw, sigma_air_liq, radius_liquid_euler, drop_condensation, compute_tau_phase
from PyLCM.collision import collection, ws_drops_beard, ws_drops_stokes, E_H80, E_S09, gck, E_turb
from PyLCM.timestep_routine import timesteps_function
from PyLCM.widget import model_steering_input, parcel_info_input, grid_modes_input, preset_input, ablation_settings, AEROSOL_PRESETS
from Post_process.analysis import ts_analysis
from Post_process.print_plot import subplot_array_function

print('All modules loaded successfully.')

In [ ]:
def run_simulation(params, seed=42):
    """Run a PyLCM simulation programmatically.

    Parameters
    ----------
    params : dict with keys:
        label (str), T0 (K), P0 (Pa), RH (0-1), w (m/s),
        n_ptcl (int), nt (int), dt (float),
        N (list, cm^-3), mu (list, um), sigma (list, geometric std), kappa (list),
        do_condensation (bool), do_collision (bool),
        switch_kelvin (bool), switch_solute (bool),
        switch_E_constant (bool), switch_vt_simple (bool),
        switch_turb_kernel (bool), epsilon_turb (float)
    seed : int

    Returns
    -------
    dict with: label, time, T, z, P, RH, qv, qc, qr, qa, nc, nr, na,
               spectra, rc_avg, con, evp, acc, aut
    """
    p = {  # defaults
        'label': 'Run', 'T0': 293.2, 'P0': 101300.0, 'RH': 0.88, 'w': 1.0,
        'n_ptcl': 1000, 'nt': 1200, 'dt': 1.0,
        'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
        'do_condensation': True, 'do_collision': False,
        'switch_kelvin': True, 'switch_solute': True,
        'switch_E_constant': False, 'switch_vt_simple': False,
        'switch_turb_kernel': False, 'epsilon_turb': 0.0,
        'switch_kappa_koehler': True,
    }
    p.update(params)

    T0, P0, RH, w = p['T0'], p['P0'], p['RH'], p['w']
    nt, dt, n_ptcl = p['nt'], p['dt'], p['n_ptcl']
    z0, zmax = 0.0, 3000.0

    # Convert aerosol parameters
    N_raw = np.array(p['N'])
    mu_raw = np.array(p['mu'])
    sigma_raw = np.array(p['sigma'])
    kappa_list = p['kappa']

    # Remove zero-N modes
    mask = N_raw > 0
    N_raw = N_raw[mask]
    mu_raw = mu_raw[mask]
    sigma_raw = sigma_raw[mask]

    N_aero = N_raw * 1e6              # cm^-3 -> m^-3
    mu_aero = np.log(mu_raw * 1e-6)   # um -> m -> ln(m)
    sigma_aero = np.log(sigma_raw)     # geometric std -> ln(sigma)

    # Environmental profiles
    theta_init = T0 * (p0 / P0)**(r_a / cp)
    theta_profs = theta_init + 5e-3 * z_env

    # Initial moisture
    e_s0 = esatw(T0)
    q0 = RH * e_s0 / (P0 - RH * e_s0) * r_a / rv

    # Initialize particles
    np.random.seed(seed)
    T_parcel, q_parcel, particles_list = aero_init(
        'Random', n_ptcl, P0, z0, T0, q0,
        N_aero, mu_aero, sigma_aero, rho_aero, kappa_list, p['switch_kappa_koehler'])

    P_parcel, z_parcel = P0, z0
    S_lst = 0.0

    # Output arrays
    time_arr = np.zeros(nt + 1)
    T_arr = np.zeros(nt + 1); z_arr = np.zeros(nt + 1); P_arr = np.zeros(nt + 1)
    RH_arr = np.zeros(nt + 1); qv_arr = np.zeros(nt + 1)
    qc_arr = np.zeros(nt + 1); qr_arr = np.zeros(nt + 1); qa_arr = np.zeros(nt + 1)
    nc_arr = np.zeros(nt + 1); nr_arr = np.zeros(nt + 1); na_arr = np.zeros(nt + 1)
    con_arr = np.zeros(nt + 1); evp_arr = np.zeros(nt + 1)
    acc_arr = np.zeros(nt + 1); aut_arr = np.zeros(nt + 1)
    rc_avg_arr = np.zeros(nt + 1)
    spectra_all = np.zeros((nt + 1, len(rm_spec)))

    # Initial analysis
    rho_p, V_p, air_mass = parcel_rho(P_parcel, T_parcel)
    sp, qa, qc, qr, na, nc, nr, _, rc_avg, _ = ts_analysis(
        particles_list, air_mass, rm_spec, n_bins, n_ptcl)
    T_arr[0] = T_parcel; z_arr[0] = z_parcel; P_arr[0] = P_parcel
    qv_arr[0] = q_parcel; qc_arr[0] = qc; qr_arr[0] = qr; qa_arr[0] = qa
    nc_arr[0] = nc; nr_arr[0] = nr; na_arr[0] = na
    rc_avg_arr[0] = rc_avg; spectra_all[0] = sp
    RH_arr[0] = (q_parcel * P_parcel / (q_parcel + r_a / rv)) / esatw(T_parcel)

    for t in range(nt):
        tt = (t + 1) * dt
        time_arr[t + 1] = tt

        # Parcel ascent
        z_parcel, T_parcel, P_parcel = ascend_parcel(
            z_parcel, T_parcel, P_parcel, w, dt, tt, zmax, theta_profs)

        rho_p, V_p, air_mass = parcel_rho(P_parcel, T_parcel)

        ct = at = et = da = 0.0
        ac = au = pr = 0.0

        # Condensation
        if p['do_condensation']:
            particles_list, T_parcel, q_parcel, S_lst, ct, at, et, da = drop_condensation(
                particles_list, T_parcel, q_parcel, P_parcel, nt, dt, air_mass,
                S_lst, rho_aero, False, ct, at, et, da,
                p['switch_kappa_koehler'], p['switch_kelvin'], p['switch_solute'])
            con_arr[t+1] = 1e3 * ct / air_mass / dt
            evp_arr[t+1] = 1e3 * et / air_mass / dt

        # Collision
        if p['do_collision']:
            particles_list, ac, au, pr = collection(
                dt, particles_list, rho_p, rho_liq, P_parcel, T_parcel,
                ac, au, pr, False, z_parcel, zmax, w,
                p['switch_E_constant'], p['switch_vt_simple'],
                p['switch_turb_kernel'], p['epsilon_turb'])
            acc_arr[t+1] = 1e3 * ac / air_mass / dt
            aut_arr[t+1] = 1e3 * au / air_mass / dt

        # Analysis
        sp, qa, qc, qr, na, nc, nr, _, rc_avg, _ = ts_analysis(
            particles_list, air_mass, rm_spec, n_bins, len(particles_list))

        T_arr[t+1] = T_parcel; z_arr[t+1] = z_parcel; P_arr[t+1] = P_parcel
        qv_arr[t+1] = q_parcel
        qc_arr[t+1] = qc; qr_arr[t+1] = qr; qa_arr[t+1] = qa
        nc_arr[t+1] = nc; nr_arr[t+1] = nr; na_arr[t+1] = na
        rc_avg_arr[t+1] = rc_avg; spectra_all[t+1] = sp
        RH_arr[t+1] = (q_parcel * P_parcel / (q_parcel + r_a / rv)) / esatw(T_parcel)

    return {
        'label': p['label'], 'time': time_arr, 'T': T_arr, 'z': z_arr, 'P': P_arr,
        'RH': RH_arr, 'qv': qv_arr, 'qc': qc_arr, 'qr': qr_arr, 'qa': qa_arr,
        'nc': nc_arr, 'nr': nr_arr, 'na': na_arr, 'spectra': spectra_all,
        'rc_avg': rc_avg_arr, 'con': con_arr, 'evp': evp_arr,
        'acc': acc_arr, 'aut': aut_arr,
    }


def plot_comparison(results_list, title='Comparison'):
    """Overlay 1-6 simulation results on a 2x4 panel grid."""
    fig, axs = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    colors = [plt.cm.tab10(i) for i in range(len(results_list))]

    for i, r in enumerate(results_list):
        c = colors[i]
        t = r['time']
        lbl = r['label']

        axs[0,0].plot(t, r['T'] - 273.15, color=c, label=lbl)
        axs[0,0].set_ylabel('T (°C)'); axs[0,0].set_xlabel('Time (s)')

        axs[0,1].plot(t, r['RH'] * 100, color=c, label=lbl)
        axs[0,1].axhline(100, color='gray', ls='--', lw=0.5)
        axs[0,1].set_ylabel('RH (%)'); axs[0,1].set_xlabel('Time (s)')

        axs[0,2].plot(t, r['qc'], color=c, ls='-', label=f"{lbl} qc")
        axs[0,2].plot(t, r['qr'], color=c, ls='--', label=f"{lbl} qr")
        axs[0,2].set_ylabel('q (g/kg)'); axs[0,2].set_xlabel('Time (s)')

        axs[0,3].plot(t, r['nc'], color=c, ls='-', label=f"{lbl} Nc")
        axs[0,3].plot(t, r['nr'], color=c, ls='--', label=f"{lbl} Nr")
        axs[0,3].set_ylabel('N (cm$^{-3}$)'); axs[0,3].set_xlabel('Time (s)')

        nt_r = len(t) - 1
        t_mid = nt_r // 2
        spec_final = copy.deepcopy(r['spectra'][nt_r])
        spec_mid = copy.deepcopy(r['spectra'][t_mid])

        axs[1,0].plot(rm_spec * 1e6, spec_final / 1e6, color=c, label=lbl)
        axs[1,0].set_xscale('log'); axs[1,0].set_xlabel('Radius ($\\mu$m)')
        axs[1,0].set_ylabel('dN/dlogr (cm$^{-3}$)'); axs[1,0].set_title('DSD at t_final')

        axs[1,1].plot(rm_spec * 1e6, spec_mid / 1e6, color=c, label=lbl)
        axs[1,1].set_xscale('log'); axs[1,1].set_xlabel('Radius ($\\mu$m)')
        axs[1,1].set_ylabel('dN/dlogr (cm$^{-3}$)'); axs[1,1].set_title('DSD at t_mid')

        axs[1,2].plot(t, r['qc'] + r['qr'], color=c, label=lbl)
        axs[1,2].set_ylabel('LWC (g/kg)'); axs[1,2].set_xlabel('Time (s)')
        axs[1,2].set_title('Total LWC')

        window = min(60, max(1, nt_r // 20))
        if np.any(r['aut'] > 0) or np.any(r['acc'] > 0):
            aut_smooth = np.convolve(r['aut'], np.ones(window)/window, mode='same')
            acc_smooth = np.convolve(r['acc'], np.ones(window)/window, mode='same')
            axs[1,3].plot(t, aut_smooth, color=c, ls='-', label=f"{lbl} aut")
            axs[1,3].plot(t, acc_smooth, color=c, ls='--', label=f"{lbl} acc")
        axs[1,3].set_ylabel('Rate (g/kg/s)'); axs[1,3].set_xlabel('Time (s)')
        axs[1,3].set_title('Autoconv / Accretion')

    for ax in axs.flat:
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    fig.tight_layout()
    plt.show()


print('Helper functions defined: run_simulation(), plot_comparison()')

## Module 1: Adiabatic Parcel Model

When an air parcel rises adiabatically, it cools at the **dry adiabatic lapse rate**:

$$\frac{dT}{dz} = -\frac{g}{c_p} \approx -9.8 \text{ K/km}$$

The pressure decreases hydrostatically:
$$dP = -\frac{P \, g}{R_a T_{\mathrm{env}}} \, dz$$

As the parcel cools, its relative humidity increases. The **saturation vapor pressure** $e_s(T)$ decreases exponentially with temperature (Clausius-Clapeyron):

$$e_s(T) \approx 611.2 \exp\left(\frac{17.62\,(T - 273.15)}{T - 29.65}\right) \text{ Pa}$$

The height at which RH reaches 100% is the **Lifting Condensation Level (LCL)**.

In [ ]:
# Saturation vapor pressure vs temperature
T_range = np.linspace(253.15, 313.15, 200)  # -20 C to +40 C
e_s = np.array([esatw(T) for T in T_range])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_range - 273.15, e_s / 100, 'b-', lw=2)
ax.set_xlabel('Temperature ($^{\\circ}$C)', fontsize=12)
ax.set_ylabel('$e_s$ (hPa)', fontsize=12)
ax.set_title('Saturation Vapor Pressure (Flatau et al. 1992)', fontsize=13)
ax.grid(alpha=0.3)
ax.annotate(f'At 20$^{{\\circ}}$C: {esatw(293.15)/100:.1f} hPa', xy=(20, esatw(293.15)/100),
            xytext=(5, esatw(293.15)/100 + 10), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='red'), color='red')
plt.tight_layout()
plt.show()

In [ ]:
# Manual parcel ascent -- tracking T, P, RH with height
T0 = 293.2; P0 = 101300.0; RH0 = 0.88; w = 1.0; dt = 1.0
nt_ascent = 1500

# Environmental profile
theta_init = T0 * (p0 / P0)**(r_a / cp)
theta_profiles = theta_init + 5e-3 * z_env

# Initial moisture
e_s0 = esatw(T0)
q0 = RH0 * e_s0 / (P0 - RH0 * e_s0) * r_a / rv

T_p, P_p, z_p, q_p = T0, P0, 0.0, q0
z_list, T_list, P_list, RH_list = [z_p], [T_p], [P_p], [RH0]
lcl_z = None

for t in range(nt_ascent):
    tt = (t + 1) * dt
    z_p, T_p, P_p = ascend_parcel(z_p, T_p, P_p, w, dt, tt, 3000.0, theta_profiles)
    RH_p = (q_p * P_p / (q_p + r_a / rv)) / esatw(T_p)
    z_list.append(z_p); T_list.append(T_p); P_list.append(P_p); RH_list.append(RH_p)
    if lcl_z is None and RH_p >= 1.0:
        lcl_z = z_p

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
axes[0].plot(np.array(T_list) - 273.15, z_list, 'r-', lw=2)
axes[0].set_xlabel('T ($^{\\circ}$C)'); axes[0].set_ylabel('Height (m)')
axes[0].set_title('Temperature Profile')
if lcl_z: axes[0].axhline(lcl_z, color='gray', ls='--', label=f'LCL $\\approx$ {lcl_z:.0f} m')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(np.array(P_list) / 100, z_list, 'b-', lw=2)
axes[1].set_xlabel('P (hPa)'); axes[1].set_ylabel('Height (m)')
axes[1].set_title('Pressure Profile'); axes[1].grid(alpha=0.3)

axes[2].plot(np.array(RH_list) * 100, z_list, 'g-', lw=2)
axes[2].axvline(100, color='gray', ls='--')
axes[2].set_xlabel('RH (%)'); axes[2].set_ylabel('Height (m)')
axes[2].set_title('Relative Humidity')
if lcl_z: axes[2].axhline(lcl_z, color='gray', ls='--', label=f'LCL $\\approx$ {lcl_z:.0f} m')
axes[2].legend(); axes[2].grid(alpha=0.3)

fig.suptitle('Adiabatic Parcel Ascent', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'LCL height = {lcl_z:.0f} m' if lcl_z else 'LCL not reached')

The temperature decreases nearly linearly with height at the dry adiabatic lapse rate ($\sim$9.8 K/km). Meanwhile, pressure drops quasi-exponentially and RH increases until saturation is reached at the LCL.

Use the interactive widget below to explore how the updraft velocity $w$ and initial relative humidity $\mathrm{RH}_0$ affect the LCL height.

In [ ]:
# Interactive LCL explorer
import ipywidgets as widgets

@widgets.interact(w_vel=(0.5, 5.0, 0.5), RH_init=(0.50, 0.98, 0.02))
def explore_lcl(w_vel=1.0, RH_init=0.88):
    T_p, P_p, z_p = T0, P0, 0.0
    e_s0_loc = esatw(T0)
    q_p = RH_init * e_s0_loc / (P0 - RH_init * e_s0_loc) * r_a / rv

    z_lst, RH_lst = [0.0], [RH_init]
    lcl = None
    for t in range(2000):
        tt = (t+1) * dt
        z_p, T_p, P_p = ascend_parcel(z_p, T_p, P_p, w_vel, dt, tt, 3000.0, theta_profiles)
        RH_p = (q_p * P_p / (q_p + r_a / rv)) / esatw(T_p)
        z_lst.append(z_p); RH_lst.append(RH_p)
        if lcl is None and RH_p >= 1.0:
            lcl = z_p

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(np.array(RH_lst) * 100, z_lst, 'g-', lw=2)
    ax.axvline(100, color='gray', ls='--')
    if lcl: ax.axhline(lcl, color='red', ls='--', label=f'LCL $\\approx$ {lcl:.0f} m')
    ax.set_xlabel('RH (%)'); ax.set_ylabel('Height (m)')
    ax.set_title(f'w = {w_vel} m/s, RH$_0$ = {RH_init*100:.0f}%')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

### Questions — Module 1

1. From the temperature profile, calculate $dT/dz$. Does it match $-g/c_p$?
2. How does the LCL height change when you increase initial RH from 70% to 95%? Why?
3. Does the updraft velocity $w$ affect the LCL height? Why or why not?
4. Once the parcel passes the LCL, the lapse rate changes from **dry** to **moist** adiabatic. Why does the moist lapse rate differ from $-g/c_p$?

## Module 2: Aerosol & CCN Activation

### Aerosol Size Distribution

Atmospheric aerosol populations follow **lognormal** distributions:

$$n(r) = \frac{N}{\sqrt{2\pi}\,r\,\ln\sigma_g} \exp\left[-\frac{(\ln r - \ln r_g)^2}{2\,\ln^2\sigma_g}\right]$$

where $N$ is total concentration, $r_g$ is the geometric mean radius, and $\sigma_g$ is the geometric standard deviation. Real atmospheres have multiple modes (Aitken, accumulation, coarse).

### Köhler Theory

The equilibrium saturation ratio over a solution droplet is:

$$S = \frac{a}{r} - \frac{b\,r_N^3}{r^3}$$

- **Kelvin (curvature) effect** $a/r$: small droplets have higher vapor pressure due to surface tension
  $$a = \frac{2\,\sigma_{w/a}}{\rho_l\,R_v\,T}$$
- **Raoult (solute) effect** $b\,r_N^3/r^3$: dissolved solute lowers vapor pressure
  - Classical: $b = \frac{i\,\rho_s\,M_w}{\rho_l\,M_s}$
  - $\kappa$-Köhler (Petters & Kreidenweis 2007): $b = \kappa$

The **critical supersaturation** $S_c$ occurs at the peak of the Köhler curve.

In [ ]:
# Lognormal aerosol size distribution -- single mode
r_plot = np.logspace(-9, -4, 500)  # m

# Maritime accumulation mode: mu=0.05 um, sigma_g=2.0, N=100 cm^-3
mu_ln = np.log(0.05e-6)    # ln(meters)
sigma_ln = np.log(2.0)     # ln(geometric std)

pdf = lognormal_pdf(r_plot, mu_ln, sigma_ln)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_plot * 1e6, 100.0 * pdf * r_plot, 'b-', lw=2)  # dN/dlogr = N * pdf * r
ax.set_xscale('log')
ax.set_xlabel('Radius ($\\mu$m)', fontsize=12)
ax.set_ylabel('dN/dlog(r) (cm$^{-3}$)', fontsize=12)
ax.set_title('Maritime Accumulation Mode: $r_g$ = 0.05 $\\mu$m, $\\sigma_g$ = 2.0', fontsize=13)
ax.axvline(0.05, color='red', ls='--', alpha=0.7, label='$r_g$ = 0.05 $\\mu$m')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Multi-modal aerosol distribution (Maritime example)
preset = AEROSOL_PRESETS['Maritime']
r_plot = np.logspace(-9, -4, 500)

fig, ax = plt.subplots(figsize=(9, 5))
total_dn = np.zeros_like(r_plot)
mode_names = ['Mode 1 (Aitken)', 'Mode 2 (Accumulation)', 'Mode 3 (Coarse)', 'Mode 4']
mode_colors = ['blue', 'green', 'red', 'purple']

for k in range(4):
    N_k = preset['N'][k]
    if N_k <= 0:
        continue
    mu_k = np.log(preset['mu'][k] * 1e-6)
    sigma_k = np.log(preset['sigma'][k])
    pdf_k = lognormal_pdf(r_plot, mu_k, sigma_k)
    dn_k = N_k * pdf_k * r_plot
    total_dn += dn_k
    ax.plot(r_plot * 1e6, dn_k, color=mode_colors[k], ls='--',
            label=f"{mode_names[k]}: N={N_k}, $r_g$={preset['mu'][k]} $\\mu$m")

ax.plot(r_plot * 1e6, total_dn, 'k-', lw=2.5, label='Total')
ax.set_xscale('log')
ax.set_xlabel('Radius ($\\mu$m)', fontsize=12)
ax.set_ylabel('dN/dlog(r) (cm$^{-3}$)', fontsize=12)
ax.set_title('Maritime Aerosol Size Distribution', fontsize=13)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Compare all three aerosol presets
r_plot = np.logspace(-9, -4, 500)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, preset) in zip(axes, AEROSOL_PRESETS.items()):
    total_dn = np.zeros_like(r_plot)
    for k in range(4):
        N_k = preset['N'][k]
        if N_k <= 0:
            continue
        mu_k = np.log(preset['mu'][k] * 1e-6)
        sigma_k = np.log(preset['sigma'][k])
        pdf_k = lognormal_pdf(r_plot, mu_k, sigma_k)
        dn_k = N_k * pdf_k * r_plot
        total_dn += dn_k
        ax.plot(r_plot * 1e6, dn_k, ls='--', alpha=0.6)
    ax.plot(r_plot * 1e6, total_dn, 'k-', lw=2.5, label='Total')
    N_total = sum(n for n in preset['N'] if n > 0)
    ax.set_xscale('log')
    ax.set_title(f'{name} ($N_{{total}}$ = {N_total:.0f} cm$^{{-3}}$)', fontsize=13)
    ax.set_xlabel('Radius ($\\mu$m)'); ax.grid(alpha=0.3); ax.legend()

axes[0].set_ylabel('dN/dlog(r) (cm$^{-3}$)')
fig.suptitle('Aerosol Presets Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Köhler Curves

The Köhler curve gives the equilibrium supersaturation as a function of droplet radius for a given dry aerosol size $r_N$ and hygroscopicity $\kappa$. Its peak defines the **critical supersaturation** $S_c$: droplets that exceed this radius grow spontaneously ("activate").

In [ ]:
# Kohler curves for different dry aerosol radii
T_k = 293.15  # K
afactor = 2.0 * sigma_air_liq(T_k) / (rho_liq * rv * T_k)

fig, ax = plt.subplots(figsize=(9, 6))
r_N_list = [0.01e-6, 0.02e-6, 0.05e-6, 0.1e-6, 0.2e-6]  # dry radii in meters
colors_k = plt.cm.viridis(np.linspace(0.1, 0.9, len(r_N_list)))

for r_N, col in zip(r_N_list, colors_k):
    kappa = 1.0  # maritime kappa
    r_drop = np.logspace(np.log10(r_N * 1.01), -3, 500)  # droplet radii
    S_kelvin = afactor / r_drop
    S_raoult = kappa * r_N**3 / r_drop**3
    S_total = S_kelvin - S_raoult

    # Find critical supersaturation
    idx_max = np.argmax(S_total)
    S_crit = S_total[idx_max] * 100  # in %
    r_crit = r_drop[idx_max]

    ax.plot(r_drop * 1e6, S_total * 100, color=col, lw=2,
            label=f'$r_N$ = {r_N*1e6:.2f} $\\mu$m ($S_c$ = {S_crit:.3f}%)')
    ax.plot(r_crit * 1e6, S_crit, 'o', color=col, ms=8)

ax.axhline(0, color='gray', ls='--', lw=0.5)
ax.set_xscale('log')
ax.set_xlabel('Droplet Radius ($\\mu$m)', fontsize=12)
ax.set_ylabel('Supersaturation S (%)', fontsize=12)
ax.set_title('Köhler Curves ($\\kappa$ = 1.0, T = 20$^{\\circ}$C)', fontsize=13)
ax.set_ylim(-0.5, 1.0)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Interactive Kohler explorer
@widgets.interact(kappa=(0.1, 2.0, 0.1), r_N_um=(0.01, 0.5, 0.01))
def kohler_explorer(kappa=1.0, r_N_um=0.05):
    r_N = r_N_um * 1e-6
    T_k = 293.15
    afactor = 2.0 * sigma_air_liq(T_k) / (rho_liq * rv * T_k)
    r_drop = np.logspace(np.log10(r_N * 1.01), -3, 500)

    S_kelvin = afactor / r_drop
    S_raoult = kappa * r_N**3 / r_drop**3
    S_total = S_kelvin - S_raoult

    idx_max = np.argmax(S_total)
    S_crit = S_total[idx_max] * 100
    r_crit = r_drop[idx_max]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(r_drop * 1e6, S_kelvin * 100, 'r--', label='Kelvin (curvature)', alpha=0.7)
    ax.plot(r_drop * 1e6, -S_raoult * 100, 'b--', label='Raoult (solute)', alpha=0.7)
    ax.plot(r_drop * 1e6, S_total * 100, 'k-', lw=2.5, label='Köhler curve')
    ax.plot(r_crit * 1e6, S_crit, 'ro', ms=10,
            label=f'$S_c$ = {S_crit:.3f}% at r = {r_crit*1e6:.2f} $\\mu$m')
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.set_xscale('log'); ax.set_ylim(-1, 1.5)
    ax.set_xlabel('Droplet Radius ($\\mu$m)'); ax.set_ylabel('S (%)')
    ax.set_title(f'Köhler Curve: $\\kappa$ = {kappa:.1f}, $r_N$ = {r_N_um:.2f} $\\mu$m')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# Equilibrium radius from Kohler theory
T_eq = 293.15
r_aero_range = np.logspace(-8, -6, 50)  # dry radii
kappas = [0.3, 0.6, 1.0]
RH_values = [0.90, 0.95, 0.99]

fig, axes = plt.subplots(1, len(RH_values), figsize=(15, 5), sharey=True)
for ax, RH_val in zip(axes, RH_values):
    S_eq = RH_val - 1.0  # subsaturated
    for kap in kappas:
        r_eq_arr = np.array([r_equi(S_eq, T_eq, r_a_val, rho_aero, True, kap)
                             for r_a_val in r_aero_range])
        ax.plot(r_aero_range * 1e6, r_eq_arr * 1e6, lw=2, label=f'$\\kappa$ = {kap}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Dry Aerosol Radius ($\\mu$m)')
    ax.set_title(f'RH = {RH_val*100:.0f}%')
    ax.legend(); ax.grid(alpha=0.3)
    ax.plot([1e-2, 1], [1e-2, 1], 'k--', alpha=0.3)

axes[0].set_ylabel('Equilibrium Wet Radius ($\\mu$m)')
fig.suptitle('Equilibrium Droplet Radius from Köhler Theory', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Visualize the initialized superdroplet population
n_ptcl_vis = 500
T0_v, P0_v, RH_v = 293.2, 101300.0, 0.88
e_s_v = esatw(T0_v)
q0_v = RH_v * e_s_v / (P0_v - RH_v * e_s_v) * r_a / rv

# Maritime preset
N_v = np.array([100.0, 20.0]) * 1e6
mu_v = np.log(np.array([0.05, 0.25]) * 1e-6)
sigma_v = np.log(np.array([2.0, 1.8]))

np.random.seed(42)
_, _, ptcls = aero_init('Random', n_ptcl_vis, P0_v, 0.0, T0_v, q0_v,
                         N_v, mu_v, sigma_v, rho_aero, [1.0, 1.2, 1.6, 1.6], True)

radii = np.array([(p.M / (p.A * 4/3 * np.pi * rho_liq))**(1/3) for p in ptcls])
weights = np.array([p.A for p in ptcls])
rho_vis = P0_v / (r_a * T0_v)
air_mass_vis = 100.0 / rho_vis * rho_vis  # = 100 kg

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sc = ax1.scatter(np.arange(len(radii)), radii * 1e6, c=weights, cmap='viridis', s=5, alpha=0.7)
plt.colorbar(sc, ax=ax1, label='Weighting factor A')
ax1.set_yscale('log')
ax1.set_xlabel('Superdroplet Index'); ax1.set_ylabel('Radius ($\\mu$m)')
ax1.set_title('Initial Superdroplet Radii')
ax1.axhline(1.0, color='red', ls='--', alpha=0.5, label='Activation threshold (1 $\\mu$m)')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.hist(radii * 1e6, bins=np.logspace(-3, 1, 50), weights=weights / air_mass_vis / 1e6)
ax2.set_xscale('log')
ax2.set_xlabel('Radius ($\\mu$m)'); ax2.set_ylabel('N (cm$^{-3}$)')
ax2.set_title('Weighted Size Distribution')
ax2.grid(alpha=0.3)

fig.suptitle(f'aero_init() Output: {n_ptcl_vis} Superdroplets (Maritime)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Total N = {sum(weights) / air_mass_vis / 1e6:.1f} cm^-3')

### The Superdroplet Concept

In Lagrangian Cloud Models, we don't track every individual droplet (there are $\sim 10^8$ per cm$^3$!). Instead, we use **superdroplets** — computational particles that each represent many real droplets.

Each superdroplet $i$ carries:
- **Weighting factor** $A_i$: the number of real droplets it represents
- **Total mass** $M_i = A_i \times m_i$ where $m_i$ is the individual droplet mass
- **Aerosol mass** $Ns_i$: the dissolved solute mass
- **Hygroscopicity** $\kappa_i$

The total number concentration is: $N = \sum_i A_i / V_{\text{parcel}}$

This approach makes it computationally feasible to simulate stochastic processes like collision-coalescence.

### Questions — Module 2

1. For a maritime aerosol with $r_N$ = 0.05 $\mu$m and $\kappa$ = 1.0, what is the critical supersaturation $S_c$?
2. How does increasing $\kappa$ affect $S_c$? Explain physically.
3. Compare the total number concentration between Maritime ($\sim$120 cm$^{-3}$) and Continental ($\sim$6100 cm$^{-3}$). Which produces more cloud droplets, and why might more CCN lead to *less* rain?
4. What role does the geometric standard deviation $\sigma_g$ play in determining the fraction of aerosols that activate?

## Module 3: Condensational Growth

### The Diffusional Growth Equation

Once a droplet activates, it grows by vapor diffusion:

$$\frac{dr^2}{dt} = 2G \left(S - \frac{a}{r} + \frac{b\,r_N^3}{r^3}\right)$$

where $G$ is the **growth factor** combining heat and vapor diffusion:

$$\frac{1}{G} = \frac{\rho_l R_v T}{e_s D_v} + \frac{\rho_l L_v}{k_a T}\left(\frac{L_v}{R_v T} - 1\right)$$

Key features of condensational growth:
- Growth rate $dr/dt \propto 1/r$ — small droplets grow faster
- This creates a **narrowing** size distribution
- The **supersaturation budget** controls everything: rising air creates $S > 0$, condensation depletes it

In [ ]:
# Growth factor G as a function of temperature
T_range_G = np.linspace(263.15, 303.15, 100)  # -10 C to 30 C
G_values = []

for T_g in T_range_G:
    e_s_g = esatw(T_g)
    thermal_conductivity = 7.94048e-5 * T_g + 0.00227011
    diff_coeff = 0.211e-4 * (T_g / 273.15)**1.94 * (101325.0 / 101300.0)
    G = 1.0 / (rho_liq * rv * T_g / (e_s_g * diff_coeff) +
               (l_v / (rv * T_g) - 1.0) * rho_liq * l_v / (thermal_conductivity * T_g))
    G_values.append(G)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_range_G - 273.15, np.array(G_values) * 1e10, 'b-', lw=2)
ax.set_xlabel('Temperature ($^{\\circ}$C)', fontsize=12)
ax.set_ylabel('G ($\\times 10^{-10}$ m$^2$/s)', fontsize=12)
ax.set_title('Condensational Growth Factor G(T)', fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Single droplet growth trajectory
T_sd = 280.0  # K, typical cloud base temperature
P_sd = 90000.0  # Pa
supersat_vals = [0.001, 0.003, 0.005, 0.01]  # 0.1% to 1% supersaturation
r_init = 1.0e-6  # 1 um initial radius
r_N_sd = 0.05e-6  # dry aerosol radius
dt_sd = 0.1  # fine timestep for accuracy
nt_sd = 6000  # 600 seconds

e_s_sd = esatw(T_sd)
thermal_conductivity = 7.94048e-5 * T_sd + 0.00227011
diff_coeff = 0.211e-4 * (T_sd / 273.15)**1.94 * (101325.0 / P_sd)
G_pre = 1.0 / (rho_liq * rv * T_sd / (e_s_sd * diff_coeff) +
               (l_v / (rv * T_sd) - 1.0) * rho_liq * l_v / (thermal_conductivity * T_sd))
alpha = 0.036
r0 = diff_coeff / alpha * np.sqrt(2.0 * np.pi / (rv * T_sd)) / \
     (1.0 + diff_coeff * l_v**2 * e_s_sd / (thermal_conductivity * rv**2 * T_sd**3))
afactor_sd = 2.0 * sigma_air_liq(T_sd) / (rho_liq * rv * T_sd)
bfactor_sd = 1.0  # kappa

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors_ss = plt.cm.viridis(np.linspace(0.1, 0.9, len(supersat_vals)))

for ss, col in zip(supersat_vals, colors_ss):
    r = r_init
    t_arr, r_arr, r2_arr = [0], [r * 1e6], [r**2 * 1e12]
    for step in range(nt_sd):
        r = radius_liquid_euler(r, dt_sd, r0, G_pre, ss, 1.0,
                                afactor_sd, bfactor_sd, r_N_sd, 0.0, 0.0)
        t_arr.append((step + 1) * dt_sd)
        r_arr.append(r * 1e6)
        r2_arr.append(r**2 * 1e12)
    ax1.plot(t_arr, r_arr, color=col, lw=2, label=f'S = {ss*100:.1f}%')
    ax2.plot(t_arr, r2_arr, color=col, lw=2, label=f'S = {ss*100:.1f}%')

ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Radius ($\\mu$m)')
ax1.set_title('r(t)'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('r$^2$ ($\\mu$m$^2$)')
ax2.set_title('r$^2$(t) — linear growth confirms diffusion'); ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Single-Droplet Growth (Euler Integration)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Ablation study: effect of Kelvin and Raoult terms on droplet growth
r_init_ab = 0.5e-6  # sub-micron: where Kelvin/Raoult matter most
ss_ab = 0.003

cases = [
    ('Full Köhler', afactor_sd, bfactor_sd),
    ('No Kelvin (a=0)', 0.0, bfactor_sd),
    ('No Raoult (b=0)', afactor_sd, 0.0),
    ('Neither (a=b=0)', 0.0, 0.0),
]

fig, ax = plt.subplots(figsize=(9, 6))
for label, af, bf in cases:
    r = r_init_ab
    t_list, r_list = [0], [r * 1e6]
    for step in range(nt_sd):
        r = radius_liquid_euler(r, dt_sd, r0, G_pre, ss_ab, 1.0, af, bf, r_N_sd, 0.0, 0.0)
        t_list.append((step+1) * dt_sd)
        r_list.append(r * 1e6)
    ax.plot(t_list, r_list, lw=2, label=label)

ax.set_xlabel('Time (s)', fontsize=12); ax.set_ylabel('Radius ($\\mu$m)', fontsize=12)
ax.set_title(f'Kelvin/Raoult Ablation (S = {ss_ab*100:.1f}%, $r_0$ = {r_init_ab*1e6:.1f} $\\mu$m)',
             fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Condensation-only simulation (Maritime, 500 particles, 1200 steps)
result_cond = run_simulation({
    'label': 'Condensation only',
    'n_ptcl': 500, 'nt': 1200,
    'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
    'do_condensation': True, 'do_collision': False,
})
print('Simulation complete!')

In [ ]:
# Condensation-only results: 6-panel plot
fig, axs = plt.subplots(2, 3, figsize=(16, 9))
r = result_cond; t = r['time']

axs[0,0].plot(t, r['T'] - 273.15, 'r-', lw=2)
axs[0,0].set_xlabel('Time (s)'); axs[0,0].set_ylabel('T ($^{\\circ}$C)')
axs[0,0].set_title('Temperature')

axs[0,1].plot(t, r['RH'] * 100, 'g-', lw=2)
axs[0,1].axhline(100, color='gray', ls='--'); axs[0,1].set_xlabel('Time (s)')
axs[0,1].set_ylabel('RH (%)'); axs[0,1].set_title('Relative Humidity')

axs[0,2].plot(t, r['qc'], 'b-', lw=2, label='$q_c$')
axs[0,2].plot(t, r['qr'], 'r--', lw=2, label='$q_r$')
axs[0,2].set_xlabel('Time (s)'); axs[0,2].set_ylabel('q (g/kg)'); axs[0,2].legend()
axs[0,2].set_title('Mixing Ratios')

axs[1,0].plot(t, r['nc'], 'b-', lw=2, label='$N_c$')
axs[1,0].plot(t, r['na'], 'gray', lw=2, label='$N_a$')
axs[1,0].set_xlabel('Time (s)'); axs[1,0].set_ylabel('N (cm$^{-3}$)'); axs[1,0].legend()
axs[1,0].set_title('Number Concentrations')

# DSD evolution
nt_c = len(t) - 1
times_show = [0, nt_c//4, nt_c//2, 3*nt_c//4, nt_c]
colors_dsd = plt.cm.viridis(np.linspace(0.1, 0.9, len(times_show)))
for ti, col in zip(times_show, colors_dsd):
    spec_i = r['spectra'][ti]
    axs[1,1].plot(rm_spec * 1e6, spec_i / 1e6, color=col, label=f't = {t[ti]:.0f} s')
axs[1,1].set_xscale('log'); axs[1,1].set_xlabel('Radius ($\\mu$m)')
axs[1,1].set_ylabel('dN/dlogr (cm$^{-3}$)'); axs[1,1].legend(fontsize=8)
axs[1,1].set_title('DSD Evolution')

# Supersaturation
S_arr = r['RH'] - 1.0
axs[1,2].plot(t, S_arr * 100, 'purple', lw=2)
axs[1,2].axhline(0, color='gray', ls='--')
axs[1,2].set_xlabel('Time (s)'); axs[1,2].set_ylabel('S (%)')
axs[1,2].set_title('Supersaturation')

for ax in axs.flat:
    ax.grid(alpha=0.3)
fig.suptitle('Condensation-Only Simulation (Maritime)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Compare different initial RH values
results_rh = []
for rh_val in [0.80, 0.88, 0.95]:
    results_rh.append(run_simulation({
        'label': f'RH_0 = {rh_val*100:.0f}%',
        'RH': rh_val, 'n_ptcl': 500, 'nt': 1200,
        'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
        'do_condensation': True, 'do_collision': False,
    }))

plot_comparison(results_rh, 'Effect of Initial Relative Humidity')

### The Supersaturation Budget

The supersaturation $S$ in a rising parcel is controlled by two competing processes:

1. **Adiabatic cooling** increases $S$ (source): As the parcel rises, $T$ decreases and $e_s$ drops faster than $e_a$
2. **Condensation** decreases $S$ (sink): As vapor condenses onto droplets, $e_a$ decreases

$$\frac{dS}{dt} \approx \alpha w - \beta \frac{dq_c}{dt}$$

where $\alpha$ depends on thermodynamic parameters and $\beta$ on the condensation rate.

- $S$ rises until condensation catches up with the source $\rightarrow$ **maximum supersaturation $S_{\max}$**
- After $S_{\max}$, the sink dominates and $S$ slowly decreases
- $S_{\max}$ determines how many CCN activate $\rightarrow$ sets $N_c$

### Questions — Module 3

1. From the $r^2(t)$ plot, estimate the growth factor $G$ using $r^2(t) = r_0^2 + 2G S \cdot t$.
2. At what height (or time) does $S_{\max}$ occur? How does this relate to the LCL?
3. In the condensation-only run, does $q_r$ ever become nonzero? Why or why not?
4. How does increasing aerosol number $N$ affect $S_{\max}$? (Think about the condensation sink.)

## Module 4: Collision-Coalescence

### The Collection Kernel

Two droplets collide and coalesce with rate:

$$K(r_1, r_2) = \pi(r_1 + r_2)^2 \, |v_1 - v_2| \, E_{\text{coll}}(r_1, r_2) \, E_{\text{coal}}(r_1, r_2)$$

where:
- $\pi(r_1 + r_2)^2$ is the geometric cross-section
- $|v_1 - v_2|$ is the differential terminal velocity
- $E_{\text{coll}}$ is the **collision efficiency** (Hall 1980) — accounts for droplet aerodynamics
- $E_{\text{coal}}$ is the **coalescence efficiency** (Straub et al. 2009) — probability of merging after contact

### Terminal Velocity

Droplet fall speed depends on size regime:
- **Small** ($d < 19\,\mu$m): Stokes drag $v_t \propto r^2$
- **Medium** ($19 < d < 1070\,\mu$m): Beard (1976) empirical fit
- **Large** ($d > 1070\,\mu$m): shape effects (large drops deform)

In [ ]:
# Terminal velocity: Beard (1976) vs Stokes
r_range = np.logspace(-6, -2.5, 200)  # 1 um to ~3 mm
T_vt, P_vt = 283.15, 90000.0
rho_p_vt = P_vt / (r_a * T_vt)

vt_beard = np.array([ws_drops_beard(r, rho_p_vt, rho_liq, P_vt, T_vt) for r in r_range])
vt_stokes = np.array([ws_drops_stokes(r, rho_p_vt, rho_liq) for r in r_range])

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(r_range * 1e6, vt_beard, 'b-', lw=2.5, label='Beard (1976)')
ax.plot(r_range * 1e6, vt_stokes, 'r--', lw=2, label='Stokes drag')
ax.set_xscale('log'); ax.set_yscale('log')
ax.axvline(9.5, color='gray', ls=':', alpha=0.5, label='d = 19 $\\mu$m')
ax.axvline(535, color='gray', ls='-.', alpha=0.5, label='d = 1070 $\\mu$m')
ax.set_xlabel('Radius ($\\mu$m)', fontsize=12)
ax.set_ylabel('Terminal Velocity (m/s)', fontsize=12)
ax.set_title('Droplet Terminal Velocity', fontsize=13)
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

In [ ]:
# Hall (1980) collision efficiency
r1_range = np.logspace(np.log10(5e-6), np.log10(400e-6), 80)
r2_range = np.logspace(np.log10(5e-6), np.log10(400e-6), 80)
E_map = np.zeros((len(r1_range), len(r2_range)))

for i, r1 in enumerate(r1_range):
    for j, r2 in enumerate(r2_range):
        E_map[i, j] = E_H80(r1, r2)

fig, ax = plt.subplots(figsize=(8, 7))
pcm = ax.pcolormesh(r1_range * 1e6, r2_range * 1e6, E_map.T,
                     cmap='viridis', vmin=0, vmax=1, shading='auto')
plt.colorbar(pcm, ax=ax, label='Collision Efficiency $E_{coll}$')
ax.set_xlabel('$r_1$ ($\\mu$m)', fontsize=12)
ax.set_ylabel('$r_2$ ($\\mu$m)', fontsize=12)
ax.set_title('Hall (1980) Collision Efficiency', fontsize=13)
ax.set_xscale('log'); ax.set_yscale('log')
ax.plot([5, 400], [5, 400], 'r--', alpha=0.5, label='$r_1 = r_2$')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Straub (2009) coalescence efficiency
r_collector = 100e-6  # 100 um collector
r_small_range = np.logspace(np.log10(5e-6), np.log10(95e-6), 50)
T_ce = 283.15
rho_p_ce = 90000.0 / (r_a * T_ce)

E_coal_arr = []
for r_s in r_small_range:
    vt1 = ws_drops_beard(r_collector, rho_p_ce, rho_liq, 90000.0, T_ce)
    vt2 = ws_drops_beard(r_s, rho_p_ce, rho_liq, 90000.0, T_ce)
    v_rel = abs(vt1 - vt2)
    E_coal_arr.append(E_S09(r_collector, r_s, v_rel, rho_liq, T_ce))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_small_range * 1e6, E_coal_arr, 'b-', lw=2)
ax.set_xlabel('Small Droplet Radius ($\\mu$m)', fontsize=12)
ax.set_ylabel('Coalescence Efficiency $E_{coal}$', fontsize=12)
ax.set_title(f'Straub (2009): Collector $r$ = {r_collector*1e6:.0f} $\\mu$m', fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Full collection kernel K(r1, r2)
r_K = np.logspace(np.log10(10e-6), np.log10(300e-6), 60)
K_map = np.zeros((len(r_K), len(r_K)))
T_K, P_K = 283.15, 90000.0
rho_K = P_K / (r_a * T_K)

for i, r1 in enumerate(r_K):
    for j, r2 in enumerate(r_K):
        vt1 = ws_drops_beard(r1, rho_K, rho_liq, P_K, T_K)
        vt2 = ws_drops_beard(r2, rho_K, rho_liq, P_K, T_K)
        v_r = abs(vt1 - vt2)
        E_c = E_H80(r1, r2)
        E_co = E_S09(r1, r2, v_r, rho_liq, T_K)
        K_map[i, j] = np.pi * (r1 + r2)**2 * v_r * E_c * E_co

fig, ax = plt.subplots(figsize=(8, 7))
pcm = ax.pcolormesh(r_K * 1e6, r_K * 1e6, np.log10(K_map.T + 1e-30),
                     cmap='hot_r', shading='auto')
plt.colorbar(pcm, ax=ax, label='log$_{10}$(K) (m$^3$/s)')
ax.set_xlabel('$r_1$ ($\\mu$m)', fontsize=12)
ax.set_ylabel('$r_2$ ($\\mu$m)', fontsize=12)
ax.set_title('Full Collection Kernel $K(r_1, r_2)$', fontsize=13)
ax.set_xscale('log'); ax.set_yscale('log')
plt.tight_layout(); plt.show()

### Stochastic Collection Algorithm (Shima et al. 2009)

In the LCM, collisions are handled stochastically between superdroplet pairs:

1. **Shuffle** the superdroplet list randomly
2. **Pair** droplets: split the list in half and pair each element
3. For each pair, compute the **collision probability**:
   $$p = \frac{A_{\max} \cdot K(r_1, r_2)}{V_{\text{parcel}}} \cdot \Delta t \cdot \frac{N(N-1)}{N_{\text{half}} \cdot 2}$$
4. Draw a random number $x \in [0,1]$: if $p > x$, collision occurs
5. **Update** the superdroplets:
   - If $A_1 = A_2$: halve weights, combine individual masses
   - If $A_1 \neq A_2$: smaller-A particle gains mass from larger-A

In [ ]:
# Condensation + Collision vs Condensation-only
result_coll = run_simulation({
    'label': 'Cond + Collision',
    'n_ptcl': 1000, 'nt': 1800,
    'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
    'do_condensation': True, 'do_collision': True,
})

result_cond_ref = run_simulation({
    'label': 'Cond only',
    'n_ptcl': 1000, 'nt': 1800,
    'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
    'do_condensation': True, 'do_collision': False,
})

plot_comparison([result_cond_ref, result_coll], 'Condensation Only vs. Condensation + Collision')

In [ ]:
# DSD evolution: condensation vs condensation+collision at key times
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
snap_times = [600, 1200, 1800]  # seconds

for ax, t_snap in zip(axes, snap_times):
    idx = t_snap  # dt=1, so index = time

    spec_cond = result_cond_ref['spectra'][idx]
    spec_coll = result_coll['spectra'][idx]

    ax.plot(rm_spec * 1e6, spec_cond / 1e6, 'b-', lw=2, label='Cond only')
    ax.plot(rm_spec * 1e6, spec_coll / 1e6, 'r-', lw=2, label='Cond + Coll')
    ax.set_xscale('log')
    ax.set_xlabel('Radius ($\\mu$m)')
    ax.set_ylabel('dN/dlogr (cm$^{-3}$)')
    ax.set_title(f't = {t_snap} s')
    ax.axvline(25, color='gray', ls='--', alpha=0.5, label='Rain threshold (25 $\\mu$m)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle('DSD Broadening by Collision-Coalescence', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Questions — Module 4

1. Why is the collision efficiency $E$ near zero for two small droplets (both < 10 $\mu$m)?
2. What is the "autoconversion bottleneck"? Why is it hard for cloud droplets to grow past $\sim$20 $\mu$m by condensation alone?
3. How does adding collision-coalescence change the DSD shape compared to condensation-only?
4. At what time does the rain mixing ratio $q_r$ first become significant? What process triggers this transition?

## Module 5: Full Warm Rain Simulation

Now we bring everything together using the full PyLCM widget interface. This module uses the same simulation pipeline as the interactive notebook (`PyLCM_edu.ipynb`), but with all the physics we've learned.

The simulation proceeds:
1. **Initialize** aerosol population $\rightarrow$ superdroplets
2. **Time loop**: parcel ascent $\rightarrow$ condensation $\rightarrow$ collision $\rightarrow$ analysis
3. **Visualize** 8-panel dashboard: RH, T, mixing ratios, number concentrations, DSD evolution, condensation rates, collision rates

In [ ]:
# Model steering inputs
dt_widget, nt_widget, Condensation_widget, Collision_widget, switch_sedi_removal, n_particles_widget, max_z_widget = model_steering_input()

In [ ]:
# Parcel initial conditions
T_widget, P_widget, RH_widget, w_widget, z_widget = parcel_info_input()

In [ ]:
# Aerosol mode parameters and preset selector
gridwidget = grid_modes_input()
preset_widget = preset_input(gridwidget)

In [ ]:
# Ablation Lab settings
kelvin_w, solute_w, E_const_w, vt_simple_w, turb_kernel_w, epsilon_w, adaptive_dt_w = ablation_settings()

# Ascending mode and display settings
import ipywidgets as widgets
ascending_mode_widget = widgets.Dropdown(
    options=['linear', 'sine', 'in_cloud_oscillation'], value='linear',
    description='Ascent mode:')
mode_displaytype_widget = widgets.Dropdown(
    options=['text_fast', 'graphics'], value='text_fast',
    description='Display:')
mode_aero_init_widget = widgets.Dropdown(
    options=['Random', 'Weighting_factor'], value='Random',
    description='Init mode:')
kohler_activation_radius = False
switch_kappa_koehler = True
display(ascending_mode_widget, mode_displaytype_widget, mode_aero_init_widget)

In [ ]:
# Create environmental profiles (required for parcel ascent)
qv_profiles, theta_profiles_env, z_env_local = create_env_profiles(
    T_widget.value,
    RH_widget.value * esatw(T_widget.value) / (P_widget.value - RH_widget.value * esatw(T_widget.value)) * r_a / rv,
    z_widget.value, P_widget.value, 'Stable')

In [ ]:
%%time
# Run the full warm rain simulation
results_full = timesteps_function(
    n_particles_widget, P_widget, RH_widget, T_widget, w_widget, nt_widget, dt_widget,
    rm_spec, ascending_mode_widget, mode_displaytype_widget, z_widget, max_z_widget,
    Condensation_widget, Collision_widget, mode_aero_init_widget, gridwidget,
    kohler_activation_radius, switch_kappa_koehler, switch_sedi_removal.value,
    0.0, False, qv_profiles, theta_profiles_env, 0, 0,
    kelvin_w.value, solute_w.value, E_const_w.value, vt_simple_w.value,
    turb_kernel_w.value, epsilon_w.value, adaptive_dt_w.value
)
print('Simulation complete!')

In [ ]:
# 8-panel dashboard
nt_full, dt_full = results_full[0], results_full[1]
import ipywidgets as widgets
increment_w = widgets.IntSlider(value=60, min=1, max=300, description='DSD interval:')
droplet_mode_w = widgets.Dropdown(
    options=['Total', 'Droplet mode', 'Aerosol mode'], value='Total',
    description='Mode:')

subplot_array_function('time-series', dt_full, nt_full, rm_spec,
    results_full[7], results_full[8], results_full[9],
    results_full[10], results_full[11], results_full[12],
    results_full[3], results_full[4], results_full[5], results_full[6],
    results_full[13],
    increment_w,
    results_full[14], results_full[15], results_full[16], results_full[17],
    results_full[18], results_full[19],
    results_full[22],
    droplet_mode_w)

### How to Read the 8-Panel Dashboard

| Panel | What It Shows | What to Look For |
|-------|---------------|------------------|
| **RH & $q_v$** | Relative humidity and vapor mixing ratio | RH reaching 100% marks cloud base |
| **T & z** | Temperature and height evolution | Cooling rate transition at LCL |
| **$q_a, q_c, q_r$** | Aerosol, cloud, rain mixing ratios | Rain onset time, peak cloud water |
| **$N_a, N_c, N_r$** | Number concentrations | Activation spike, collision depletion |
| **DSD contour** | Full size spectrum over time | Cloud-rain transition, DSD broadening |
| **DSD snapshots** | Spectra at selected times | Bimodality development |
| **Cond/Evap rates** | Mass flux from condensation | Peak near LCL, decrease with height |
| **Auto/Accretion** | Collision process rates | Autoconversion first, then accretion dominates |

### Questions — Module 5

1. At approximately what time (or height) does rain onset occur? What is $q_r$ at that point?
2. Describe the three stages of the cloud lifecycle visible in the dashboard: activation, growth, precipitation.
3. Does the DSD become bimodal? At what time do you first see a clear rain mode?
4. Try the Maritime vs Continental presets. How does the total number concentration affect the time to rain onset?
5. Enable the turbulent collision kernel. Does it speed up rain formation?

## Part 1 Summary

| Module | Key Concept | Key Equation |
|--------|-------------|-------------|
| 1. Parcel Model | LCL from adiabatic cooling | $dT/dz = -g/c_p$ |
| 2. Aerosol/CCN | Köhler theory, $\kappa$-parameterization | $S = a/r - b r_N^3/r^3$ |
| 3. Condensation | Diffusional growth, S budget | $dr^2/dt = 2G(S - a/r + br_N^3/r^3)$ |
| 4. Collision | Collection kernel, SCA | $K = \pi(r_1+r_2)^2|v_1-v_2|E_{coll}E_{coal}$ |
| 5. Full Simulation | Warm rain lifecycle | All combined |

**Next**: Part 2 applies these concepts in guided numerical experiments.